In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="torch")
warnings.simplefilter("ignore")

In [2]:
# hyperparameters
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_size = 1
hidden_size = 64
num_layers = 2
output_size = 1

In [3]:
# get the model
class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size, dropout=0.3):
        super(LSTMClassifier, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # LSTM layer
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        
        # Fully connected output layer
        self.fc = nn.Linear(hidden_size, output_size)
        
        # Sigmoid activation function for binary classification
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        # Initialize hidden state and cell state with zeros
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        
        # Forward propagate through LSTM
        out, _ = self.lstm(x, (h0, c0))
        
        # Pass the output of the last time step through the fully connected layer
        out = self.fc(out[:, -1, :])  # Extract the last time step output
        
        # Apply sigmoid activation to get probability
        return self.sigmoid(out)
model = LSTMClassifier(input_size, hidden_size, num_layers, output_size).to(device)
checkpoint_path = 'checkpoint.pth'

if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print("checkpoint loaded")
else:
    raise FileNotFoundError("checkpoint doesnt exist")


checkpoint loaded


In [4]:
model.eval()

LSTMClassifier(
  (lstm): LSTM(1, 64, num_layers=2, batch_first=True, dropout=0.3)
  (fc): Linear(in_features=64, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

In [5]:
max_len = 573
def pad_sequence(mags, max_len):
    padded = torch.zeros((1, max_len, 1)).to(device)
    length = min(len(mags), max_len)
    padded[0, :length, 0] = mags[:length]
    return padded

In [6]:
sample_curve_path = r"caitlinbegbie CORONA melina gaia_mainseq\0.88jds_new_Gaia19ekz.csv"
sample_curve = pd.read_csv(sample_curve_path)
sample_curve = sample_curve.sort_values(by="avg Mag")

# Ensure the column contains only valid numeric data
mags = pd.to_numeric(sample_curve["avg Mag"], errors='coerce').dropna()

# Check if the column is empty after cleaning
if mags.empty:
    raise ValueError("The 'avg Mag' column is empty or contains only invalid data.")

# Convert to PyTorch tensor and add batch and feature dimensions
mags_tensor = torch.tensor(mags.to_numpy(), dtype=torch.float32).to(device)
print(mags_tensor.size())
mags_tensor = pad_sequence(mags_tensor, max_len)
print(mags_tensor.size())
with torch.no_grad():
    probability = model(mags_tensor)

#print(f"Probability of being an RCB star: {probability:.4f}")
print(probability)

torch.Size([171])
torch.Size([1, 573, 1])
tensor([[0.0309]], device='cuda:0')


In [7]:
sample_curve_path = r"rcb data\zenodo rcbs 2\0.88jds noisey2_lc_wise-toi-185.csv"
sample_curve = pd.read_csv(sample_curve_path)
sample_curve = sample_curve.sort_values(by="mags")

# Ensure the column contains only valid numeric data
mags = pd.to_numeric(sample_curve["mags"], errors='coerce').dropna()

# Check if the column is empty after cleaning
if mags.empty:
    raise ValueError("The 'avg Mag' column is empty or contains only invalid data.")

# Convert to PyTorch tensor and add batch and feature dimensions
mags_tensor = torch.tensor(mags.to_numpy(), dtype=torch.float32).to(device)
print(mags_tensor.size())
mags_tensor = pad_sequence(mags_tensor, max_len)
print(mags_tensor.size())
with torch.no_grad():
    probability = model(mags_tensor)

#print(f"Probability of being an RCB star: {probability:.4f}")
print(probability)

torch.Size([332])
torch.Size([1, 573, 1])
tensor([[1.0000]], device='cuda:0')
